In [ ]:
tasks = ['ner', 'clause', 'risk']
print(f"{'Task':<10} {'Train':>8} {'Eval':>8} {'Total':>8}")
print('-' * 38)
for task in tasks:
    train = load_jsonl(PROCESSED_DIR / f'{task}_train.jsonl')
    eval_ = load_jsonl(PROCESSED_DIR / f'{task}_eval.jsonl')
    print(f'{task:<10} {len(train):>8,} {len(eval_):>8,} {len(train)+len(eval_):>8,}')
print('\nAll datasets ready for training.')

## 7. Processed Dataset Summary

In [ ]:
def verify_annotations(samples, name, spot_n=5):
    print(f'\n=== {name} — Annotation Spot Check ===')
    errors = 0
    for sample in samples[:50]:
        text = sample['text']
        for ent in sample['entities']:
            extracted = text[ent['start']:ent['end']]
            if extracted != ent['text']:
                print(f"  MISMATCH: expected '{ent['text']}', got '{extracted}'")
                errors += 1
    status = 'OK — all positions correct' if errors == 0 else f'{errors} mismatches found'
    print(f'  Result: {status} (checked first 50 samples)')

    print(f'\n  Example spans from sample[0]:')
    for ent in samples[0]['entities'][:spot_n]:
        span = samples[0]['text'][ent['start']:ent['end']]
        ok = 'OK' if span == ent['text'] else 'MISMATCH'
        print(f"    [{ok}] {ent['label']:20s} ({ent['start']:4d},{ent['end']:4d}) = '{span}'")

verify_annotations(loan_docs, 'Loan Docs')
verify_annotations(kyc_forms, 'KYC Forms')

## 6. Entity Annotation Quality Check

In [ ]:
print('=== NER INSTRUCTION FORMAT (Loan Doc) ===')
print(loan_docs[2]['instruction_formatted'][:900])
print('\n' + '='*60)
print('\n=== RISK SCORING INSTRUCTION FORMAT (Credit Memo) ===')
print(credit_memos[1]['instruction_formatted'][:900])
print('\n' + '='*60)
print('\n=== NER INSTRUCTION FORMAT (KYC Form) ===')
print(kyc_forms[0]['instruction_formatted'][:900])

## 5. Instruction Format Examples

In [ ]:
def word_count(text):
    return len(text.split())

datasets = {
    'Loan Docs (NER)':    [word_count(d['text']) for d in loan_docs],
    'Credit Memos (Risk)':[word_count(d['text']) for d in credit_memos],
    'KYC Forms (NER)':    [word_count(d['text']) for d in kyc_forms],
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, lengths) in zip(axes, datasets.items()):
    mean = sum(lengths) / len(lengths)
    ax.hist(lengths, bins=30, color='teal', edgecolor='white')
    ax.axvline(mean, color='red', linestyle='--', label=f'Mean: {mean:.0f}')
    ax.set_title(name)
    ax.set_xlabel('Word Count')
    ax.set_ylabel('Frequency')
    ax.legend()
plt.suptitle('Text Length Distributions', y=1.02)
plt.tight_layout()
plt.show()

print('\nSummary:')
for name, lengths in datasets.items():
    print(f"  {name}: min={min(lengths)}, max={max(lengths)}, mean={sum(lengths)/len(lengths):.0f}")

## 4. Text Length Analysis

In [ ]:
# Entity type frequency — Loan Docs vs KYC Forms
all_loan_labels = [e['label'] for d in loan_docs for e in d['entities']]
all_kyc_labels  = [e['label'] for d in kyc_forms for e in d['entities']]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, counts, title in [
    (axes[0], Counter(all_loan_labels), 'Loan Doc Entity Frequencies'),
    (axes[1], Counter(all_kyc_labels),  'KYC Form Entity Frequencies'),
]:
    lbls, vals = zip(*counts.most_common())
    bars = ax.bar(lbls, vals, color='slateblue')
    ax.bar_label(bars, padding=2, fontsize=9)
    ax.set_xticklabels(lbls, rotation=35, ha='right')
    ax.set_ylabel('Frequency')
    ax.set_title(title)
plt.tight_layout()
plt.show()

In [ ]:
# Risk level distribution
risk_counts = Counter(d['risk_level'] for d in credit_memos)
colors = {'LOW': '#2ecc71', 'MEDIUM': '#f39c12', 'HIGH': '#e74c3c'}

fig, ax = plt.subplots(figsize=(6, 4))
for level in ['LOW', 'MEDIUM', 'HIGH']:
    count = risk_counts.get(level, 0)
    ax.bar(level, count, color=colors[level])
    ax.text(level, count + 2, str(count), ha='center', fontweight='bold')
ax.set_ylabel('Count')
ax.set_title('Credit Memo — Risk Level Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Loan type distribution
loan_type_counts = Counter(d['metadata']['loan_type'] for d in loan_docs)
labels, values = zip(*loan_type_counts.most_common())

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.barh(labels, values, color='steelblue')
ax.bar_label(bars, padding=3)
ax.set_xlabel('Count')
ax.set_title('Loan Agreement — Loan Type Distribution')
plt.tight_layout()
plt.show()

## 3. Distribution Charts

In [ ]:
print('=== KYC FORM SAMPLE ===')
print(kyc_forms[0]['text'][:800])
print('\nEntities found:')
for e in kyc_forms[0]['entities']:
    print(f"  [{e['label']}] {e['text']}")

In [ ]:
print('=== CREDIT MEMO SAMPLE ===')
print(credit_memos[0]['text'][:800])
print(f"\nRisk Level:     {credit_memos[0]['risk_level']}")
print(f"Recommendation: {credit_memos[0]['recommendation']}")

In [ ]:
print('=== LOAN AGREEMENT SAMPLE ===')
print(loan_docs[0]['text'][:800])
print('\nEntities found:')
for e in loan_docs[0]['entities']:
    print(f"  [{e['label']}] {e['text']}")

## 2. Sample Documents

In [ ]:
loan_docs    = load_jsonl(RAW_DIR / 'loan_docs.jsonl')
credit_memos = load_jsonl(RAW_DIR / 'credit_memos.jsonl')
kyc_forms    = load_jsonl(RAW_DIR / 'kyc_forms.jsonl')

print(f'Loan docs:     {len(loan_docs):,}')
print(f'Credit memos:  {len(credit_memos):,}')
print(f'KYC forms:     {len(kyc_forms):,}')
print(f'Total:         {len(loan_docs)+len(credit_memos)+len(kyc_forms):,}')

## 1. Load Datasets

In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

print('Setup complete.')

# BankDocAI — Data Exploration

Explores three synthetic datasets:
- **Loan Agreements** (NER task)
- **Credit Memos** (Risk Scoring task)
- **KYC Forms** (NER task)

Covers: distributions, text lengths, entity annotation quality, instruction format examples.